# 📊 03 — Models vs Baselines

Final results on the tasks of `docs/model_plan.md`. **T1 contagion** (main task): after coin *i* shocks at hour *t*,
does coin *j* shock within *k* hours? **T2**: does a coin shock in the next hour? **T3**: realized volatility over
the next 1/6/24 hours.

1. Run registry: what was run, at which commit.
2. T1 contagion: TGN vs the independent cascade and the base rate (validation and test, 5 seeds).
3. Seed ensemble.
4. What the temporal graph memory adds (ablation).
5. Significance: day-block bootstrap on the test events.
6. Over- or underfitting? Train vs validation lift.
7. What else we tried (development runs, validation only).
8. 1h vs 2h robustness.
9. T2 and T3: LSTM (no graph) vs simple baselines.

**Protocol.** Every model is scored on the same events: sample bars only (7-day embargo; label window inside the
split). All tuning used validation; the test period was scored once per final run (`evaluate_test: true`). AUPRC must
be read against the positive rate (= the AUPRC of random guessing).

In [1]:
import sys, json
from pathlib import Path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch  # keep torch before pandas (Windows DLL order)
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from cryptograph.analysis.evaluation import block_bootstrap_auprc_diff
from cryptograph.network.cascade import IndependentCascade
from cryptograph.training.metrics import auprc, auroc

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_dark"
V2 = "../data/processed/v2"
RUNS = Path("../runs")
K = [1, 3, 6, 12, 24]
FINAL = {"TGN (1h)": "tgn_1h_final_v2", "TGN without memory (1h)": "tgn_1h_nomem_final_v3", "TGN (2h)": "tgn_2h_final_v2"}

runs = pd.read_csv(RUNS / "results.csv")
runs = runs[runs["status"] == "ok"]
cas = pd.read_csv("../reports/cascade_baseline.csv")
base = pd.concat([pd.read_csv(f"../reports/hourly_baselines_{r}.csv") for r in ("1h", "2h")])

def metrics_of(exp):
    out = []                                          # run dir = {YYYYmmdd-HHMMSS}_{experiment}_{seed}_{commit}
    for d in sorted(RUNS.iterdir()):
        if not d.is_dir() or not (d / "metrics.json").exists() or len(d.name) < 17:
            continue
        name, seed, _ = d.name[16:].rsplit("_", 2)
        if name == exp:
            m = json.loads((d / "metrics.json").read_text())
            m["run_dir"], m["seed"] = d, int(seed)
            out.append(m)
    return out

reg = runs.groupby("experiment").agg(runs=("run_id", "size"), seeds=("seed", lambda s: sorted(set(s.tolist()))),
                                     commit=("commit", lambda c: ", ".join(sorted(set(map(str, c))))),
                                     uncommitted=("dirty", "any"))
display(reg.loc[[e for e in reg.index if not e.startswith("smoke")]])

,runs,seeds,commit,uncommitted
experiment,,,,
lstm_1h_rv_dev,1,[0],d9ebc62,False
lstm_1h_rv_final,5,"[0, 1, 2, 3, 4]",e46fe2c,False
lstm_1h_shock_dev,1,[0],d9ebc62,False
lstm_1h_shock_final,5,"[0, 1, 2, 3, 4]",e46fe2c,False
tgn_1h_ctx_dev,1,[0],8f5d566,True
tgn_1h_ctx_nomem_dev,1,[0],8f5d566,True
tgn_1h_ctx_reg_a_dev,1,[0],8f5d566,True
tgn_1h_ctx_reg_b_dev,1,[0],8f5d566,True
tgn_1h_dev,1,[0],b331a89,True


*What this shows:* Every run behind this notebook, with the git commit it ran at. All **final** runs (`*_final*`) ran on a clean
commit (`uncommitted = False`) and scored the test period once. Development runs (`*_dev`) were scored on validation
only; some carry uncommitted changes because they tested code that was committed afterwards (`ef514d4`). `tgn_1h_final`
(v1) is kept for the record but was under-trained (see section 6).

## 1. T1 contagion: TGN vs the independent cascade

In [2]:
def model_rows(exp, label, res):
    rows = []
    ms = metrics_of(exp)
    for split in ("val", "test"):
        for k in K:
            v = np.array([m.get(f"{split}_auprc_k{k}h", np.nan) for m in ms], float)
            if np.isfinite(v).any():
                rows.append({"model": label, "resolution": res, "split": split, "k_hours": k,
                             "auprc": np.nanmean(v), "std": np.nanstd(v, ddof=1), "seeds": int(np.isfinite(v).sum())})
    return rows

rows = []
for label, exp in FINAL.items():
    rows += model_rows(exp, label, "2h" if "2h" in exp else "1h")
for res in ("1h", "2h"):
    for name, lab in (("independent cascade", "independent cascade"), ("receiver base rate", "base rate (random)")):
        c = cas[(cas["resolution"] == res) & (cas["model"] == name) & (cas["split"] != "train")]
        rows += [{"model": f"{lab} ({res})", "resolution": res, "split": r.split, "k_hours": r.k_hours,
                  "auprc": r.positive_rate if lab.startswith("base") else r.auprc, "std": 0.0, "seeds": 0}
                 for r in c.itertuples()]
t1 = pd.DataFrame(rows)
r1 = t1[t1["resolution"] == "1h"]
fig = px.line(r1, x="k_hours", y="auprc", color="model", error_y="std", facet_col="split", markers=True, log_x=True,
              category_orders={"split": ["val", "test"]},
              title="T1 contagion, 1h: AUPRC by horizon (TGN: mean ± std over 5 seeds)")
fig.update_xaxes(tickvals=K, title="k (hours)")
fig.update_layout(height=430)
fig.show()
tab = r1.pivot_table(index=["split", "k_hours"], columns="model", values="auprc")
tab["TGN / cascade"] = tab["TGN (1h)"] / tab["independent cascade (1h)"]
tab["TGN / random"] = tab["TGN (1h)"] / tab["base rate (random) (1h)"]
display(tab.round(3).loc[["val", "test"]])

model          TGN (1h)  TGN without memory (1h)  base rate (random) (1h)  \
split k_hours                                                               
val   1           0.153                    0.146                    0.086   
      3           0.257                    0.251                    0.174   
      6           0.362                    0.346                    0.269   
      12          0.482                    0.463                    0.397   
      24          0.652                    0.628                    0.580   
test  1           0.205                    0.208                    0.105   
      3           0.352                    0.344                    0.206   
      6           0.464                    0.454                    0.310   
      12          0.588                    0.576                    0.447   
      24          0.739                    0.729                    0.627   

model          independent cascade (1h)  TGN / cascade  TGN / random  
split k_hours                                                         
val   1                           0.117          1.302         1.779  
      3                           0.212          1.211         1.483  
      6                           0.307          1.178         1.344  
      12                          0.411          1.171         1.213  
      24                          0.579          1.127         1.125  
test  1                           0.159          1.284         1.948  
      3                           0.279          1.260         1.711  
      6                           0.381          1.218         1.496  
      12                          0.507          1.162         1.317  
      24                          0.672          1.099         1.179

*What this shows:* The TGN beats the independent cascade at every horizon, on validation and on test. On test at *k* = 1 hour its
AUPRC is **0.205** vs 0.159 for the cascade and 0.105 for random guessing: about **2× better than random and 1.3×
better than the cascade**. The advantage is largest for the next few hours and narrows at 24 hours (0.739 vs 0.672),
where many coins shock anyway. Test scores are higher than validation for every model, including the cascade, because
2024–26 had more shocks and stronger co-movement than 2022–23; that is a property of the period, not of the models.

## 2. Seed ensemble

Averaging the predicted probabilities of the 5 seeds needs no tuning and usually helps a little.

In [3]:
def preds(exp, split):
    ps = [torch.load(m["run_dir"] / f"predictions_{split}.pt", weights_only=False) for m in metrics_of(exp)
          if (m["run_dir"] / f"predictions_{split}.pt").exists()]
    pos = ps[0]["position"].numpy()
    assert all(np.array_equal(p["position"].numpy(), pos) for p in ps), "seeds must score the same events"
    return ps

ens_rows = []
for split in ("val", "test"):
    ps = preds("tgn_1h_final_v2", split)
    prob = torch.stack([p["prob"].float() for p in ps]).mean(0).numpy()
    y, m = ps[0]["y"].numpy(), ps[0]["mask"].numpy()
    for i, k in enumerate(K):
        single = np.mean([auprc(y[:, i], p["prob"].float().numpy()[:, i], m[:, i]) for p in ps])
        ens_rows.append({"split": split, "k_hours": k, "seed mean": single, "5-seed ensemble": auprc(y[:, i], prob[:, i], m[:, i]),
                         "ensemble AUROC": auroc(y[:, i], prob[:, i], m[:, i])})
ens = pd.DataFrame(ens_rows).set_index(["split", "k_hours"])
ens["gain"] = ens["5-seed ensemble"] - ens["seed mean"]
display(ens.round(4))

seed mean  5-seed ensemble  ensemble AUROC    gain
split k_hours                                                    
val   1           0.1527           0.1532          0.6429  0.0005
      3           0.2572           0.2577          0.6242  0.0005
      6           0.3616           0.3621          0.6157  0.0005
      12          0.4814           0.4820          0.5997  0.0005
      24          0.6522           0.6528          0.5914  0.0006
test  1           0.2045           0.2055          0.6857  0.0011
      3           0.3517           0.3531          0.6598  0.0014
      6           0.4641           0.4654          0.6486  0.0013
      12          0.5882           0.5893          0.6408  0.0011
      24          0.7386           0.7395          0.6358  0.0010

*What this shows:* Averaging the 5 seeds adds about +0.001 AUPRC on test: a small, free gain. The seeds already agree closely (std
≈ 0.003), so there is little variance left to average away. AUROC is 0.64–0.69: the model ranks events clearly better than
chance (0.5) but far from perfectly, as expected for hourly, largely news-driven shocks.

## 3. What the temporal graph memory adds (ablation)

The same model with memory and neighbour attention switched off (`use_memory: false`): the head still sees the
node features at *t* and the event message. v3 adds `min_epochs: 30` because in v2 four seeds stopped on a noisy
epoch-1 peak; the full TGN is unaffected by that rule (its seeds stopped after epoch 55).

In [4]:
ab = r1[r1["model"].isin(["TGN (1h)", "TGN without memory (1h)", "independent cascade (1h)"])]
piv = ab.pivot_table(index=["split", "k_hours"], columns="model", values="auprc")
piv["memory adds"] = piv["TGN (1h)"] - piv["TGN without memory (1h)"]
piv["no-memory model over cascade"] = piv["TGN without memory (1h)"] - piv["independent cascade (1h)"]
display(piv.round(4).loc[["val", "test"]])
per_seed = pd.DataFrame([{"model": lab, "seed": m["seed"], "best epoch": m["best_epoch"], "val k1h": m["val_auprc_k1h"],
                          "test k1h": m["test_auprc_k1h"], "test k24h": m["test_auprc_k24h"]}
                         for lab, exp in list(FINAL.items())[:2] for m in metrics_of(exp)])
display(per_seed.round(4))

model          TGN (1h)  TGN without memory (1h)  independent cascade (1h)  \
split k_hours                                                                
val   1          0.1529                   0.1455                    0.1174   
      3          0.2573                   0.2511                    0.2125   
      6          0.3617                   0.3463                    0.3071   
      12         0.4816                   0.4634                    0.4112   
      24         0.6523                   0.6279                    0.5786   
test  1          0.2046                   0.2078                    0.1594   
      3          0.3519                   0.3437                    0.2792   
      6          0.4643                   0.4544                    0.3811   
      12         0.5884                   0.5760                    0.5065   
      24         0.7387                   0.7295                    0.6722   

model          memory adds  no-memory model over cascade  
split k_hours                                             
val   1             0.0073                        0.0281  
      3             0.0062                        0.0386  
      6             0.0154                        0.0392  
      12            0.0182                        0.0522  
      24            0.0245                        0.0492  
test  1            -0.0031                        0.0484  
      3             0.0082                        0.0645  
      6             0.0100                        0.0732  
      12            0.0125                        0.0694  
      24            0.0093                        0.0573

,model,seed,best epoch,val k1h,test k1h,test k24h
0,TGN (1h),0,98,0.1528,0.2032,0.7387
1,TGN (1h),1,45,0.1459,0.2095,0.7409
2,TGN (1h),2,76,0.1557,0.2039,0.7378
3,TGN (1h),3,55,0.1522,0.2021,0.7382
4,TGN (1h),4,86,0.1577,0.2043,0.7381
5,TGN without memory (1h),0,46,0.1458,0.2072,0.7288
6,TGN without memory (1h),1,47,0.1454,0.2073,0.7293
7,TGN without memory (1h),2,45,0.1454,0.2087,0.7302
8,TGN without memory (1h),3,49,0.1456,0.2086,0.7295
9,TGN without memory (1h),4,59,0.1453,0.2070,0.7297


*What this shows:* With memory and neighbour attention switched off, the model is **just as good at 1 hour** (test 0.208 vs 0.205)
but **worse at 3–24 hours** (by 0.008–0.013 on test, 0.006–0.025 on validation). So the temporal graph memory helps
with slower contagion over several hours, not with the immediate next hour, where the event features and the
receiver's own state already carry the signal. Most of the gain over the cascade (+0.05 to +0.07 on test) comes from
learning on those features, which the no-memory model also has. The no-memory seeds are extremely consistent (best
epochs 45–59, std ≤ 0.001) now that early stopping cannot fire before epoch 30.

## 4. Significance: day-block bootstrap on the test events

TGN (5-seed ensemble) against the independent cascade and against the no-memory ablation, on exactly the same
test events. Whole days are resampled (1,000 times), because one crash creates hundreds of correlated events.

In [5]:
def cascade_scores(res, t, dst):
    p = torch.load(f"{V2}/{res}/panel.pt", weights_only=False)
    gi = torch.load(f"{V2}/{res}/tgn_events.pt", weights_only=False)["graph_index"]
    sh, va, act = (p[k].numpy()[:, gi] for k in ("shock_today", "shock_today_mask", "active"))
    bh = p["bar_hours"]
    kb = [max(1, round(k / bh)) for k in K]
    ic = IndependentCascade.fit(sh, va, p["sample_mask"]["train"].numpy(), sorted(set(kb)), 10.0)
    src_all = sh & va & act
    return np.stack([ic.cascade(k, src_all[t], dst) for k in kb], axis=1), bh

def boot(exp_a, other, res="1h", n_boot=1000):
    pa = preds(exp_a, "test")
    a = torch.stack([p["prob"].float() for p in pa]).mean(0).numpy()
    y, m, t, dst = pa[0]["y"].numpy(), pa[0]["mask"].numpy(), pa[0]["t"].numpy(), pa[0]["dst"].numpy()
    if other == "cascade":
        b, bh = cascade_scores(res, t, dst)
    else:
        pb = preds(other, "test")
        assert np.array_equal(pb[0]["position"].numpy(), pa[0]["position"].numpy())
        b, bh = torch.stack([p["prob"].float() for p in pb]).mean(0).numpy(), (2 if res == "2h" else 1)
    day = (t * bh) // 24
    rows = []
    for i, k in enumerate(K):
        mk = m[:, i]
        r = block_bootstrap_auprc_diff(y[mk, i], a[mk, i], b[mk, i], day[mk], n_boot=n_boot, seed=i)
        rows.append({"k_hours": k, "AUPRC model": auprc(y[mk, i], a[mk, i]), f"AUPRC {other}": auprc(y[mk, i], b[mk, i]), **r})
    return pd.DataFrame(rows).set_index("k_hours")

print("TGN (1h, 5-seed ensemble) vs independent cascade:")
bt_cas = boot("tgn_1h_final_v2", "cascade")
display(bt_cas.round(4))
print("TGN (1h) vs TGN without memory (both 5-seed ensembles):")
bt_mem = boot("tgn_1h_final_v2", "tgn_1h_nomem_final_v3")
display(bt_mem.round(4))

TGN (1h, 5-seed ensemble) vs independent cascade:


,AUPRC model,AUPRC cascade,diff,ci_low,ci_high,p_one_sided,n_blocks,n_events
k_hours,,,,,,,,
1,0.2055,0.1594,0.0462,0.0192,0.0738,0.0,971,795787
3,0.3531,0.2792,0.0739,0.0395,0.1140,0.0,971,795787
6,0.4654,0.3811,0.0842,0.0511,0.1156,0.0,971,795787
12,0.5893,0.5065,0.0828,0.0583,0.1066,0.0,971,795773
24,0.7395,0.6722,0.0674,0.0494,0.0837,0.0,971,795765


TGN (1h) vs TGN without memory (both 5-seed ensembles):


,AUPRC model,AUPRC tgn_1h_nomem_final_v3,diff,ci_low,ci_high,p_one_sided,n_blocks,n_events
k_hours,,,,,,,,
1,0.2055,0.2081,-0.0025,-0.0144,0.0096,0.608,971,795787
3,0.3531,0.3439,0.0092,-0.0016,0.0206,0.049,971,795787
6,0.4654,0.4545,0.0109,0.0001,0.0219,0.025,971,795787
12,0.5893,0.5760,0.0133,0.0039,0.0234,0.002,971,795773
24,0.7395,0.7296,0.0099,0.0033,0.0172,0.004,971,795765


*What this shows:* **TGN vs cascade: significant at every horizon.** The 95% day-block intervals are all above zero (e.g. +0.046
[0.019, 0.074] at 1 hour, +0.084 [0.051, 0.116] at 6 hours) and no resample out of 1,000 favoured the cascade.
**TGN vs no memory: significant only at 6–24 hours** (+0.011 to +0.013, p ≤ 0.025); at 1 hour the difference is −0.003
[−0.014, +0.010], i.e. no difference, and at 3 hours it is borderline (p = 0.049, interval touching zero). Resampling whole
days (971 of them) is what makes these intervals honest: one crash produces hundreds of correlated events.

## 5. Over- or underfitting? Train vs validation lift

Lift = AUPRC ÷ positive rate. Some drop from train (2020–22) to validation (2022–23) is expected because the market
changed; the independent cascade (~900 parameters, cannot memorise) measures that part. A larger drop than the
cascade's means the model learned train-period specifics (overfitting); a model that does no better than the cascade
on train would be underfitting.

In [6]:
fit = []
cs = cas[(cas["resolution"] == "1h") & (cas["model"] == "independent cascade")].set_index(["split", "k_hours"])
for k in (1, 3, 24):
    lt = cs.loc[("train", k), "auprc"] / cs.loc[("train", k), "positive_rate"]
    lv = cs.loc[("val", k), "auprc"] / cs.loc[("val", k), "positive_rate"]
    fit.append({"model": "independent cascade", "k_hours": k, "train lift": lt, "val lift": lv, "ratio": lt / lv})
diag = {"TGN (lr-schedule dev run, same architecture)": "tgn_1h_lrsched_dev", "TGN + context (dev)": "tgn_1h_ctx_dev",
        "TGN without memory (v3, 5 seeds)": "tgn_1h_nomem_final_v3", "TGN + context, heavy regularization (dev)": "tgn_1h_ctx_reg_a_dev",
        "TGN + context, smaller model (dev)": "tgn_1h_ctx_reg_b_dev"}
for lab, exp in diag.items():
    ms = [m for m in metrics_of(exp) if "train_auprc_k1h" in m]
    for k in (1, 3, 24):
        if not ms:
            continue
        lt = np.mean([m[f"train_auprc_k{k}h"] / m[f"train_posrate_k{k}h"] for m in ms])
        lv = np.mean([m[f"val_auprc_k{k}h"] / m[f"val_posrate_k{k}h"] for m in ms])
        fit.append({"model": lab, "k_hours": k, "train lift": lt, "val lift": lv, "ratio": lt / lv})
fit = pd.DataFrame(fit)
display(fit.pivot_table(index="model", columns="k_hours", values=["val lift", "ratio"]).round(2))

ratio             val lift        \
k_hours                                         1     3     24       1     3    
model                                                                           
TGN (lr-schedule dev run, same architecture)  1.50  1.40  1.11     1.76  1.50   
TGN + context (dev)                           1.51  1.41  1.11     1.76  1.48   
TGN + context, heavy regularization (dev)     1.27  1.26  1.11     1.67  1.47   
TGN + context, smaller model (dev)            1.37  1.31  1.13     1.55  1.44   
TGN without memory (v3, 5 seeds)              1.34  1.32  1.13     1.69  1.45   
independent cascade                           1.09  1.11  1.13     1.37  1.22   

                                                    
k_hours                                         24  
model                                               
TGN (lr-schedule dev run, same architecture)  1.13  
TGN + context (dev)                           1.14  
TGN + context, heavy regularization (dev)     1.10  
TGN + context, smaller model (dev)            1.10  
TGN without memory (v3, 5 seeds)              1.08  
independent cascade                           1.00

*What this shows:* The cascade's train→validation drop (ratio ≈ 1.1) is the part explained by the market changing. Every learned
model drops more at short horizons: about 1.5 for the TGN, 1.34 for the no-memory model, 1.27 with heavy
regularization. So **the TGN does overfit the training period somewhat at 1–3 hours**, while at 24 hours all models
are close to the cascade (1.08–1.14), i.e. no overfitting there. The overfitting is controlled, not eliminated: early
stopping picks the epoch with the best validation score, and stronger regularization or a smaller model narrowed the
gap only by losing validation accuracy (section 6). Nothing is underfitting: every model beats the cascade on train
and on validation.

## 6. What else we tried (development runs, validation only)

In [7]:
dev = {"v1 final settings (30 epochs, patience 4), 5 seeds": "tgn_1h_final",
       "v2 settings (100 epochs, patience 10), 5 seeds": "tgn_1h_final_v2",
       "+ learning-rate schedule": "tgn_1h_lrsched_dev", "+ event context features": "tgn_1h_ctx_dev",
       "+ context, no memory": "tgn_1h_ctx_nomem_dev", "+ context, heavy regularization": "tgn_1h_ctx_reg_a_dev",
       "+ context, smaller model": "tgn_1h_ctx_reg_b_dev"}
rows = []
for lab, exp in dev.items():
    ms = metrics_of(exp)
    if ms:
        rows.append({"variant": lab, "runs": len(ms), "best epoch": ", ".join(str(m["best_epoch"]) for m in ms),
                     **{f"val k{k}h": np.mean([m[f"val_auprc_k{k}h"] for m in ms]) for k in (1, 3, 24)}})
display(pd.DataFrame(rows).set_index("variant").round(4))

,runs,best epoch,val k1h,val k3h,val k24h
variant,,,,,
"v1 final settings (30 epochs, patience 4), 5 seeds",5,"15, 5, 6, 29, 19",0.1384,0.2518,0.6455
"v2 settings (100 epochs, patience 10), 5 seeds",5,"98, 45, 76, 55, 86",0.1529,0.2573,0.6523
+ learning-rate schedule,1,84,0.1511,0.2608,0.6578
+ event context features,1,44,0.1509,0.2570,0.6610
"+ context, no memory",1,56,0.1555,0.2617,0.6482
"+ context, heavy regularization",1,8,0.1434,0.2544,0.6389
"+ context, smaller model",1,4,0.1332,0.2499,0.6353


*What this shows:* What we tried, judged on validation only. **Training long enough mattered most**: the v1 budget (30 epochs,
patience 4) stopped while the score was still rising (0.138 → 0.153 with v2). After that, nothing moved validation
AUPRC by more than about ±0.005, which is the size of the seed-to-seed noise: a learning-rate schedule, event-context
features (market breadth, shock state of both coins), or dropping memory. Heavier regularization and a smaller model
were clearly worse. We therefore kept the simplest well-trained model (v2) as the final TGN rather than adopting a
variant that won by noise.

## 7. Robustness: 1h vs 2h

In [8]:
r2 = t1[t1["resolution"] == "2h"]
display(r2.pivot_table(index=["split", "k_hours"], columns="model", values="auprc").round(3).loc[["val", "test"]])
bt2 = boot("tgn_2h_final_v2", "cascade", res="2h")
display(bt2.round(4))

model          TGN (2h)  base rate (random) (2h)  independent cascade (2h)
split k_hours                                                             
val   1           0.085                    0.068                     0.080
      3           0.133                    0.108                     0.116
      6           0.178                    0.150                     0.152
      12          0.293                    0.241                     0.240
      24          0.498                    0.411                     0.415
test  1           0.165                    0.092                     0.148
      3           0.237                    0.145                     0.198
      6           0.299                    0.190                     0.255
      12          0.420                    0.292                     0.357
      24          0.560                    0.443                     0.490

,AUPRC model,AUPRC cascade,diff,ci_low,ci_high,p_one_sided,n_blocks,n_events
k_hours,,,,,,,,
1,0.1678,0.1476,0.0202,-0.0114,0.0578,0.102,924,410824
3,0.2405,0.1981,0.0423,0.0108,0.0813,0.004,924,410824
6,0.3026,0.2545,0.0481,0.0149,0.0831,0.003,924,410824
12,0.4247,0.3575,0.0672,0.0324,0.0996,0.000,924,410814
24,0.5639,0.4898,0.0741,0.0460,0.1012,0.000,924,410809


*What this shows:* At 2-hour resolution the TGN still beats the cascade at every horizon (test 0.168 vs 0.148 at one bar, 0.564 vs
0.490 at 24 hours), significant from 3 hours on; at one bar the interval includes zero (p = 0.10). Coarser bars blur
which coin moved first, so the advantage is smaller than at 1h: **1h is the right resolution for this task**, and the 2h
result confirms the conclusion is not an artefact of one bar size.

## 8. T2 shock risk and T3 volatility: LSTM (no graph) vs simple baselines

Final LSTM runs (`lstm_1h_shock_final`, `lstm_1h_rv_final`): 5 seeds each, settings frozen from the development runs,
test scored once. Baselines from `reports/hourly_baselines_1h.csv`, on exactly the same (hour, coin) samples.

In [9]:
b1 = base[(base["resolution"] == "1h") & base["split"].isin(["val", "test"])]
sh = metrics_of("lstm_1h_shock_final")
t2 = b1[b1["task"].str.startswith("T2")].pivot_table(index="model", columns="split", values=["AUPRC", "AUROC"])
t2.loc["LSTM (168h window, own coin only), 5 seeds"] = [np.mean([m[f"{s}_{k.lower()}"] for m in sh])
                                                         for k, s in t2.columns]
t2.loc["random (positive rate)"] = [np.mean([m[f"{s}_posrate"] for m in sh]) if k == "AUPRC" else 0.5 for k, s in t2.columns]
print(f"LSTM seed std of test AUPRC: {np.std([m['test_auprc'] for m in sh], ddof=1):.4f}; best epochs {[m['best_epoch'] for m in sh]}")
display(t2.sort_values(("AUPRC", "test"), ascending=False).round(4))

from cryptograph.data.datasets import PanelWindowDataset
from cryptograph.models.lstm import CryptoLSTM
from torch.utils.data import DataLoader

def lstm_rv_mse(run_dir, split):
    ck = torch.load(run_dir / "checkpoint.pt", weights_only=False)
    mc = ck["model_config"]
    net = CryptoLSTM(in_features=ck["n_features"], hidden_dim=mc["hidden_dim"], num_layers=mc["num_layers"],
                     out_horizons=ck["n_outputs"], dropout=mc["dropout"])
    net.load_state_dict(ck["model_state"])
    net.eval()
    ds = PanelWindowDataset(f"{V2}/1h/panel.pt", f"{V2}/1h/lstm_index_{split}.pt")
    P, Y = [], []
    with torch.no_grad():
        for b in DataLoader(ds, batch_size=8192):
            P.append(net(torch.nan_to_num(b["x"].float())))
            Y.append(b["y_rv"].float())
    P, Y = torch.cat(P).numpy(), torch.cat(Y).numpy()
    ok = np.isfinite(Y)
    return [float(np.mean(((P - Y) ** 2)[:, i][ok[:, i]])) for i in range(Y.shape[1])]

t3 = b1[b1["task"].str.startswith("T3")].pivot_table(index="model", columns=["split", "horizon_hours"], values="MSE")
rv = metrics_of("lstm_1h_rv_final")
lstm = {s: np.array([lstm_rv_mse(m["run_dir"], s) for m in rv]) for s in ("val", "test")}
t3.loc["LSTM (168h window, own coin only), 5 seeds"] = [lstm[s].mean(0)[[1, 6, 24].index(h)] for s, h in t3.columns]
display(t3.sort_values(("test", 24)).map(lambda v: f"{v:.3e}" if np.isfinite(v) else ""))
print("LSTM seed std of test MSE per horizon:", np.round(lstm["test"].std(0, ddof=1), 7))

LSTM seed std of test AUPRC: 0.0027; best epochs [13, 1, 12, 8, 6]


AUPRC           AUROC        
split                                         test     val    test     val
model                                                                     
LSTM (168h window, own coin only), 5 seeds  0.1091  0.0857  0.7056  0.6798
logistic on the 4 rules (train-fitted)      0.0954  0.0787  0.6746  0.6468
|z| now                                     0.0855  0.0758  0.6295  0.6193
market breadth now                          0.0683  0.0579  0.6081  0.6032
own shocks last 24h                         0.0649  0.0528  0.6319  0.5999
shock now                                   0.0528  0.0470  0.5524  0.5452
train base rate per coin                    0.0392  0.0368  0.5047  0.5067
random (positive rate)                      0.0387  0.0363  0.5000  0.5000

split                                            test                        \
horizon_hours                                      1          6          24   
model                                                                         
HAR-RV (train-fitted)                       5.012e-05  2.488e-05  1.681e-05   
LSTM (168h window, own coin only), 5 seeds  5.468e-05  2.783e-05  2.027e-05   
persistence (trailing 24h RV)                                     2.163e-05   
train mean per coin                         6.920e-05  5.136e-05  4.326e-05   
zero                                        1.020e-04  1.020e-04  1.020e-04   
persistence (trailing 1h RV)                8.426e-05                         
persistence (trailing 6h RV)                           3.675e-05              

split                                             val                        
horizon_hours                                      1          6          24  
model                                                                        
HAR-RV (train-fitted)                       4.021e-05  2.161e-05  1.472e-05  
LSTM (168h window, own coin only), 5 seeds  4.350e-05  2.497e-05  1.853e-05  
persistence (trailing 24h RV)                                     1.857e-05  
train mean per coin                         6.358e-05  5.462e-05  4.884e-05  
zero                                        7.529e-05  7.524e-05  7.512e-05  
persistence (trailing 1h RV)                6.804e-05                        
persistence (trailing 6h RV)                           3.257e-05

LSTM seed std of test MSE per horizon: [3.7e-06 1.1e-06 7.0e-07]


*What this shows:* **T2 (next-hour shock of one coin), final runs, 5 seeds:** the LSTM, which sees only that coin's last 168 hours,
reaches test AUPRC **0.109 ± 0.003** (AUROC 0.706): **2.8× random** (0.039) and better than every simple rule (best:
logistic regression on 4 rules, 0.095). Validation agrees (0.086 vs 0.079). **T3 (volatility):** the classic **HAR-RV**
model beats the LSTM at every horizon, on validation and test (test MSE 5.0e-5 vs 5.5e-5 at 1 hour, 1.7e-5 vs 2.0e-5 at
24 hours); the LSTM beats "same as last period" at 1 and 6 hours but only matches it at 24 hours. For forecasting how
volatile one coin will be, the simple linear model is the right tool; deep learning earns its place on the rare-event
and network questions.

## Key Takeaways

- **Contagion is partly predictable, and the TGN predicts it best.** Test AUPRC at 1 hour: TGN 0.205, independent cascade
  0.159, random 0.105; the TGN beats the cascade at every horizon (day-block bootstrap, all 95% intervals above zero).
- **The temporal graph memory matters for slower contagion.** It adds +0.011–0.013 AUPRC at 6–24 hours (significant) and
  nothing at 1 hour, where the event and node features already carry the signal.
- **Results are stable and honest.** Five seeds agree within ±0.003; tuning used validation only; test was scored once
  per final run; every model was compared on the same events.
- **Some overfitting at short horizons, controlled by early stopping.** The TGN's train→validation drop exceeds the
  market-change drop (1.5 vs 1.1 at 1 hour); regularization reduced it only at the cost of validation accuracy.
- **Training budget mattered more than architecture tweaks.** v1 was under-trained; context features, LR schedules and
  regularization did not beat the plain, well-trained TGN beyond seed noise.
- **Single-coin tasks:** the LSTM is the best next-hour shock predictor (test AUPRC 0.109, 2.8× random, above all rules);
  for volatility the classic HAR-RV model beats the LSTM at every horizon. **1h beats 2h** for contagion.

**Next:** DyGFormer through DyGLib as a second temporal graph model (notebook 03 gets a new row), then TempME
explanations of the TGN (notebook 04).